# ConformerFlow — Google Colab Pro (A100)

**Before running:** Go to `Runtime → Change runtime type → A100 GPU`

**Cell order:** Run cells top to bottom, do not skip any.

## Cell 1 — Mount Google Drive (saves checkpoints permanently)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## Cell 2 — Clone repo and install dependencies

In [ ]:
!git clone https://github.com/kumar23kan/conformerflow.git /content/conformerflow
!pip install -q biopython gemmi requests tqdm numpy einops scipy

## Cell 3 — Set paths

In [ ]:
import os, sys
os.chdir('/content/conformerflow/files')
sys.path.insert(0, '/content/conformerflow/files')
print('Working directory:', os.getcwd())

## Cell 4 — Check GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('CUDA available:', torch.cuda.is_available())

## Cell 5 — Create directories

In [ ]:
from pathlib import Path
for d in ['pdb_data/nmr', 'pdb_data/parsed_nmr', 'pdb_data/splits', 'checkpoints']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('Done')

## Cell 6 — Download NMR PDB files

`max_results=2000` downloads ~2000 PDB entries (~15–20 min). Increase to 5000 for full scale.

In [ ]:
from data.fetch_pdb import fetch_nmr_ids, batch_download
ids = fetch_nmr_ids(min_conformers=5, max_results=2000)
print(f'Found {len(ids)} NMR entries')
batch_download(ids, Path('pdb_data/nmr'), max_workers=8)

## Cell 7 — Parse NMR structures

In [ ]:
!python3 data/parse_nmr.py \
    --pdb_dir pdb_data/nmr \
    --out_dir pdb_data/parsed_nmr \
    --min_conformers 5 \
    --max_residues 300 \
    --n_workers 4

## Cell 8 — Filter and split data

In [ ]:
!python3 data/filter.py \
    --parsed_dir pdb_data/parsed_nmr \
    --splits_dir pdb_data/splits \
    --train_frac 0.8 \
    --val_frac 0.1 \
    --test_frac 0.1

## Cell 9 — Train

Key settings for A100:
- `batch_size=4` — A100 has 40GB, can handle larger batches
- `use_bf16=true` — A100 natively supports BF16, ~2x speedup
- `max_epochs=100` — full training run
- `n_gen_conformers=10` — more conformers per protein

In [ ]:
!python3 train.py \
    --config configs/base_config.yaml \
    --batch_size 4 \
    --max_epochs 100 \
    --n_gen_conformers 10 \
    --use_bf16 true \
    --max_residues 300 \
    --checkpoint_dir checkpoints \
    --val_every 200 \
    --save_every 500

## Cell 10 — Save checkpoints to Google Drive

**Run this immediately after training completes** (or any time during training to save progress).

In [ ]:
import shutil, os
from pathlib import Path

ckpt_dir = '/content/conformerflow/files/checkpoints'
drive_dir = '/content/drive/MyDrive/conformerflow_checkpoints'

if os.path.exists(ckpt_dir) and os.listdir(ckpt_dir):
    shutil.copytree(ckpt_dir, drive_dir, dirs_exist_ok=True)
    print('Checkpoints saved to Drive:')
    for f in os.listdir(drive_dir):
        size = os.path.getsize(os.path.join(drive_dir, f)) / 1e6
        print(f'  {f}  ({size:.1f} MB)')
else:
    print('No checkpoints found. Training may not have completed any epoch.')

## Cell 11 — Keep-alive (prevents Colab from timing out)

Run this in a separate browser tab or alongside training.

In [ ]:
import time
from IPython.display import display, Javascript
display(Javascript('''
function keepAlive() {
    document.querySelector('#run-all-cells-below').click();
    setTimeout(keepAlive, 60000);
}
'''))
print('Keep-alive active. Colab will not disconnect due to inactivity.')

## Cell 12 — Resume from checkpoint (if session was interrupted)

If your session was interrupted, copy checkpoint back from Drive and resume training.

In [ ]:
import shutil, os

drive_dir = '/content/drive/MyDrive/conformerflow_checkpoints'
ckpt_dir = '/content/conformerflow/files/checkpoints'

if os.path.exists(drive_dir):
    shutil.copytree(drive_dir, ckpt_dir, dirs_exist_ok=True)
    print('Checkpoints restored from Drive:')
    for f in os.listdir(ckpt_dir):
        print(f'  {f}')
else:
    print('No saved checkpoints found in Drive.')